# Sta-RU Dubbing — from a Drive folder (no YouTube)

Dubs videos you've **already put in a Google Drive folder** — never touches YouTube, so there's **no bot wall** to hit.

**Multi-account workflow:** one *host* account holds the source videos in a Drive folder and shares it; each working account adds a shortcut to that folder (Drive → *Shared with me* → *Add shortcut to Drive*), mounts its own Drive here, reads the videos, and writes the dubbed outputs to **its own** Drive. Run several accounts in parallel for more throughput.

**File naming:** each video filename must start with its catalog number (`21 - Title.mp4`, `21.mp4`, or `… - 21 - ….mp4`). Subtitles are named `{N#}-{LANG}.srt` (e.g. `21-EN.srt`, `21-ES.srt`, `21-DE.srt`).

## 1. Setup — install & clone

No YouTube preflight here: this notebook reads local files, so there's nothing to probe.

In [ ]:
import os, sys
!apt-get -qq install -y ffmpeg rubberband-cli
!pip install -q edge-tts nest-asyncio srt soundfile numpy scipy demucs deep-translator pyrubberband ipywidgets pandas faster-whisper
REPO_DIR = '/content/Sta-RU'
BRANCH = 'claude/compassionate-knuth-M2llL'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b $BRANCH https://github.com/lazy-money/sta-ru.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --quiet
_colab = os.path.join(REPO_DIR, 'colab')
if _colab not in sys.path:
    sys.path.insert(0, _colab)
print('Repo ready at', REPO_DIR, '(branch:', BRANCH, ')')
print('No YouTube here -> no bot wall, no cookies, no preflight.')


## 2. Mount Google Drive

Mount the Drive of **this** session's account. Your own files (and any folder you added a shortcut to) appear under `/content/drive/MyDrive/...`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Point to your videos & subtitles

`Videos dir` = the Drive folder with the `.mp4`s (the host's shared folder, via your shortcut, or your own). `Subtitles dir` = the folder with the `{N#}-{LANG}.srt` files (can be the same folder).

In [ ]:
import ipywidgets as W
from IPython.display import display
videos_dir_w = W.Text(value='/content/drive/MyDrive/Host Videos',
                      description='Videos dir:', layout=W.Layout(width='720px'))
srt_dir_w    = W.Text(value='/content/drive/MyDrive/Subs_IT',
                      description='Subtitles dir:', layout=W.Layout(width='720px'))
display(videos_dir_w, srt_dir_w)
print('Set both folders, then continue to Options.')
print('Tip: subtitles live in per-language folders (Subs_IT, Subs_ES, ...) -> point this at the one matching your target language.')


## 4. Options

In [ ]:
import ipywidgets as W, os
from IPython.display import display
from batch_dub_edge import DEFAULT_VOICES, VOICE_CHOICES

_drive_mounted = os.path.ismount('/content/drive')
LANG_OPTIONS = list(DEFAULT_VOICES.keys())
lang_w   = W.Dropdown(options=LANG_OPTIONS, value='EN', description='Target language:')
gender_w = W.RadioButtons(options=[('Male','M'),('Female','F')], value='M', description='Voice gender:')
def _voice_options(lang, gender):
    vs = VOICE_CHOICES.get(lang, {}).get(gender) or [DEFAULT_VOICES[lang][gender]]
    return [(v + ' (default)' if i==0 else v, v) for i, v in enumerate(vs)]
voice_custom_w = W.Dropdown(options=_voice_options('EN','M'), description='Voice:', layout=W.Layout(width='600px'))
pitch_w        = W.IntSlider(value=-5, min=-20, max=20, step=1, description='Pitch (Hz):')
range_w        = W.Text(value='all', description='Range (N#):', placeholder="'all','1-10','47'")
remove_voice_w = W.Checkbox(value=True,  description='Remove original voice (Demucs)')
dynamic_dur_w  = W.Checkbox(value=True,  description='Dynamic duration (stretch video to fit dubbing)')
skip_silent_w  = W.Checkbox(value=True,  description='Skip TTS where the original speaker is silent')
burn_subs_w    = W.Checkbox(value=False, description='Burn subtitles into the video')
multilang_w    = W.Checkbox(value=False, description='Multi-language run (dub reads Subs_<LANG> per language; Whisper sweeps the whole Dubbing folder)',
                            layout=W.Layout(width='760px'), style={'description_width': 'initial'})
hq_demucs_w    = W.Checkbox(value=False, description='Higher-quality vocal removal (mdx_extra) - slower')
allow_no_amb_w = W.Checkbox(value=False, description='Continue without ambient if Demucs fails')
ambient_gain_w = W.FloatSlider(value=3.5, min=1.0, max=8.0, step=0.25, description='Ambient gain:', readout_format='.2f')
out_mode_w   = W.RadioButtons(options=[('Save to Google Drive','drive'),('Save locally','local')],
                              value=('drive' if _drive_mounted else 'local'), description='Output:')
out_path_w   = W.Text(value=('/content/drive/MyDrive/Dubbing/EN' if _drive_mounted else '/content/output/EN'),
                      description='Output dir:', layout=W.Layout(width='600px'))
cache_path_w = W.Text(value='/tmp/sta-ru-cache', description='Cache dir:', layout=W.Layout(width='600px'))

# --- secondary 'Other lang dub' cascade (reuses each video's ambient once) ---
_sec_label = W.HTML('<b>Other lang dub</b> (optional - default voice each; each video is processed once and reused)')
sec_box = W.VBox([]); _ALL_LANGS = list(DEFAULT_VOICES.keys()); _rebuilding = {'on': False}
def _sec_opts(exclude):
    return [('— none —', None)] + [(l, l) for l in _ALL_LANGS if l not in exclude]
def chosen_secondaries():
    out = []
    for dd in sec_box.children:
        v = dd.value
        if v and v != lang_w.value and v not in out: out.append(v)
    return out
def _rebuild_cascade(_=None):
    if _rebuilding['on']: return
    _rebuilding['on'] = True
    try:
        picks = chosen_secondaries(); used = [lang_w.value] + picks; rows = []
        for i, p in enumerate(picks):
            dd = W.Dropdown(options=_sec_opts([u for u in used if u != p]), value=p,
                            description=f'Other dub {i+1}:', layout=W.Layout(width='320px'))
            dd.observe(_rebuild_cascade, names='value'); rows.append(dd)
        if [l for l in _ALL_LANGS if l not in used]:
            dd = W.Dropdown(options=_sec_opts(used), value=None,
                            description=f'Other dub {len(picks)+1}:', layout=W.Layout(width='320px'))
            dd.observe(_rebuild_cascade, names='value'); rows.append(dd)
        sec_box.children = tuple(rows)
    finally:
        _rebuilding['on'] = False
def _sync(_=None):
    lg = lang_w.value
    out_path_w.value = (f'/content/drive/MyDrive/Dubbing/{lg}' if out_mode_w.value=='drive' else f'/content/output/{lg}')
def _refresh_voices(_=None):
    voice_custom_w.options = _voice_options(lang_w.value, gender_w.value); voice_custom_w.index = 0
lang_w.observe(_sync, names='value'); out_mode_w.observe(_sync, names='value')
lang_w.observe(_refresh_voices, names='value'); gender_w.observe(_refresh_voices, names='value')
lang_w.observe(_rebuild_cascade, names='value'); _rebuild_cascade()

display(lang_w, gender_w, voice_custom_w, _sec_label, sec_box, multilang_w, pitch_w, range_w,
        remove_voice_w, dynamic_dur_w, skip_silent_w, burn_subs_w,
        hq_demucs_w, allow_no_amb_w, ambient_gain_w, out_mode_w, out_path_w, cache_path_w)
print('Pick the primary + (optional) other languages, then run the next cell.')
print('Multi-language run: OFF = one language this Colab. Dub reads the single Subtitles dir;')
print('                          Whisper (step 7) subtitles just this language folder (.../Dubbing/<LANG>).')
print('                    ON  = each language from its own Subs_<LANG> folder;')
print('                          Whisper sweeps the whole Dubbing folder, filling EVERY video missing an .srt.')
if not _drive_mounted:
    print('NOTE: Drive not mounted (step 2) - output defaulted to local.')


## 5. Scan folders & lock configuration

Lists the videos, parses each N# from the filename, and shows which `{N#}-{LANG}.srt` it found for the languages you chose.

In [ ]:
import os, re
import batch_dub
from batch_dub_edge import resolve_voice

VIDEOS_DIR = videos_dir_w.value.strip()
SRT_DIR    = srt_dir_w.value.strip()
assert os.path.isdir(VIDEOS_DIR), f'Videos dir not found: {VIDEOS_DIR}'
assert os.path.isdir(SRT_DIR), f'Subtitles dir not found: {SRT_DIR}'

_EXTS = ('.mp4','.mkv','.webm','.mov','.m4v','.avi')
vids = [os.path.join(VIDEOS_DIR, f) for f in os.listdir(VIDEOS_DIR) if f.lower().endswith(_EXTS)]
if not vids:
    raise RuntimeError(f'No video files in {VIDEOS_DIR}')

def _parse_n(path):
    name = os.path.basename(path)
    m = re.search(r' - (\d+) - ', name)            # '2020-08-05 - 95 - Title.mp4'
    if m: return int(m.group(1))
    m = re.match(r'\s*(\d+)', name)                # '95 - Title.mp4' / '95.mp4'
    if m: return int(m.group(1))
    return None

# Sort by parsed N# so processing order is 1,2,3,... (not 1,10,11,...).
# Files without a parseable N# go to the end and get skipped anyway.
_parsed = sorted(((_parse_n(p), p) for p in vids),
                 key=lambda np: (np[0] is None, np[0] or 0, os.path.basename(np[1]).lower()))

SRC_PATHS, N_OVERRIDES, _rows = [], [], []
for n, p in _parsed:
    SRC_PATHS.append(p); N_OVERRIDES.append(n); _rows.append((n, os.path.basename(p)))

CONFIG = {
    'lang':                 lang_w.value,
    'secondary_langs':      chosen_secondaries(),
    'gender':               gender_w.value,
    'voice':                voice_custom_w.value.strip() or None,
    'pitch_st':             pitch_w.value,
    'range_expr':           range_w.value,
    'remove_voice':         remove_voice_w.value,
    'dynamic_duration':     dynamic_dur_w.value,
    'skip_silent_segments': skip_silent_w.value,
    'burn_in_subs':         burn_subs_w.value,
    'demucs_model':         'mdx_extra' if hq_demucs_w.value else 'htdemucs',
    'ambient_gain':         ambient_gain_w.value,
    'allow_no_ambient':     allow_no_amb_w.value,
    'output_dir':           out_path_w.value,
    'srt_dir':              SRT_DIR,
    'cache_root':           cache_path_w.value or None,
    'multilang':            multilang_w.value,
}
if CONFIG['output_dir'].startswith('/content/drive/') and not os.path.ismount('/content/drive'):
    CONFIG['output_dir'] = f"/content/output/{CONFIG['lang']}"
    print('[NOTICE] Drive not mounted - output falls back to local:', CONFIG['output_dir'])
batch_dub.AMBIENT_GAIN = CONFIG['ambient_gain']

# Multi-language run -> read each language's SRTs from its own Subs_<LANG> folder
# (derived from the Subtitles dir: if it already points at a Subs_<LANG> folder,
# use its parent and look for siblings; otherwise treat it as the base). OFF ->
# the single Subtitles dir is used for every language (one-language-per-Colab).
CONFIG['srt_dirs'] = None
if CONFIG['multilang']:
    _all = [CONFIG['lang']] + CONFIG['secondary_langs']
    _b = os.path.basename(SRT_DIR.rstrip('/'))
    _base = os.path.dirname(SRT_DIR.rstrip('/')) if re.fullmatch(r'Subs_[A-Za-z]{2,6}', _b) else SRT_DIR
    CONFIG['srt_dirs'] = {lg: os.path.join(_base, f'Subs_{lg}') for lg in _all}

def _srt_dir_for(lg):
    return (CONFIG['srt_dirs'] or {}).get(lg, SRT_DIR)

print(f'Found {len(SRC_PATHS)} videos in {VIDEOS_DIR}')
print(f"Mode: {'MULTI-language' if CONFIG['multilang'] else 'single-language'}")
print(f"Primary:   {CONFIG['lang']}  -> {resolve_voice(CONFIG['lang'], CONFIG['gender'], CONFIG['voice'])}")
for lg in CONFIG['secondary_langs']:
    print(f"Secondary: {lg}  -> {resolve_voice(lg, CONFIG['gender'], None)} (default)")
if CONFIG['srt_dirs']:
    print("Subtitles: per-language -> " + ", ".join(f"{lg}:{d}" for lg, d in CONFIG['srt_dirs'].items()))
else:
    print(f"Subtitles: {SRT_DIR}")
print(f"Output base: {CONFIG['output_dir']}")
print()
_miss = [f for n, f in _rows if n is None]
if _miss:
    print('[WARN] no N# parsed (these will be SKIPPED - rename to start with the number):')
    for f in _miss: print('   ', f)
_langs = [CONFIG['lang']] + CONFIG['secondary_langs']
print(f"{'N#':>4}  video  ->  SRTs found")
for n, f in _rows:
    if n is None: continue
    found = [lg for lg in _langs if os.path.exists(os.path.join(_srt_dir_for(lg), f'{n}-{lg}.srt'))]
    miss  = [lg for lg in _langs if lg not in found]
    warn  = f"   [missing: {', '.join(miss)}]" if miss else ''
    print(f"{n:>4}  {f[:46]:46}  {', '.join(found) or '-'}{warn}")


## 6. Run

Each video is processed once (Demucs ambient cached); the primary language dubs first, then each *Other lang dub* reuses the same cached audio. A real TTS failure aborts that one video instead of shipping it with a silent gap.

In [ ]:
from batch_dub_edge import run_batch_multilang

results = run_batch_multilang(
    source_paths=SRC_PATHS,
    n_overrides=N_OVERRIDES,
    srt_dir=CONFIG['srt_dir'],
    srt_dirs=CONFIG.get('srt_dirs'),
    output_dir=CONFIG['output_dir'],
    primary_lang=CONFIG['lang'],
    secondary_langs=CONFIG['secondary_langs'],
    gender=CONFIG['gender'],
    voice=CONFIG['voice'],
    pitch_st=CONFIG['pitch_st'],
    translate_titles=False,
    remove_voice=CONFIG['remove_voice'],
    dynamic_duration=CONFIG['dynamic_duration'],
    skip_silent_segments=CONFIG['skip_silent_segments'],
    burn_in_subs=CONFIG['burn_in_subs'],
    cache_root=CONFIG['cache_root'],
    range_expr=CONFIG['range_expr'],
    demucs_model=CONFIG['demucs_model'],
    allow_no_ambient=CONFIG['allow_no_ambient'],
    normalize_local=False,
)


## 7. Subtitles for the dubbed videos (Whisper)

Transcribes the dubbed videos with `faster-whisper` and writes `{video}.srt` **next to each one** — the *TTS-timed* subtitles (same content as the translated `{N#}-{LANG}.srt`, but matching the dubbed audio's timing instead of the original).

**It sweeps the chosen folder and fills every video missing an `.srt`** — not just this run's, any older ones too. The next cell shows a **Whisper source** text box; the run cell after it does the work. Two ways to use this step:

- **After dubbing in this Colab:** leave the box empty — it auto-picks `…/Dubbing/<LANG>` in single-language mode, or the whole `…/Dubbing/` tree in multi-language mode.
- **Whisper-only (no dubbing run):** do steps 1 + 2, skip 3–6, paste the folder you want swept (`…/Dubbing` for every language, `…/Dubbing/DE` for just German), and run the cell below it. **Language is automatic** from each per-language subfolder name (`…/DE/…` → German); unknown folders fall back to auto-detect.

Mirrors your Windows `faster-whisper-xxl.exe` flow (`large-v2`, `int8_float16`, `temperature 0`, `beam_size 5`, `best_of 1`). `vad_filter` is on so the ambient-only gaps the dubbing leaves don't get hallucinated text — set `WHISPER_VAD = False` in the run cell for the bare Windows behavior.

In [ ]:
import ipywidgets as W, os
from IPython.display import display

# Default value: if step 5 has run, leave the box empty so the run cell auto-
# picks the right folder from CONFIG. Otherwise prefill with /content/drive/MyDrive/Dubbing
# as a helpful starting point that the user can edit before running.
_wsrc_default = '' if 'CONFIG' in globals() else '/content/drive/MyDrive/Dubbing'
whisper_source_w = W.Text(
    value=_wsrc_default,
    placeholder='/content/drive/MyDrive/Dubbing  (parent for all langs, or .../Dubbing/DE for one)',
    description='Whisper source:', layout=W.Layout(width='760px'),
    style={'description_width': 'initial'})
display(whisper_source_w)
print('Edit the folder if needed, then run the next cell.')
if 'CONFIG' in globals():
    print('Leave it empty -> the run cell uses this Colab\'s dub output folder automatically.')
else:
    print('Whisper-only flow: just type the folder Whisper should sweep, then run the next cell.')


In [ ]:
# faster-whisper — same engine as the Windows faster-whisper-xxl.exe (Purfview build)
try:
    from faster_whisper import WhisperModel
except ImportError:
    !pip install -q faster-whisper
    from faster_whisper import WhisperModel
import srt, torch
from pathlib import Path
from datetime import timedelta
from batch_dub_edge import DEFAULT_VOICES   # standalone import: lets this cell run

WHISPER_MODEL  = 'large-v2'   # same model as your Windows flow
WHISPER_VAD    = True         # skip ambient-only gaps so Whisper doesn't hallucinate text there
SKIP_EXISTING  = True         # like the PowerShell `Test-Path $srt` guard: only fill MISSING .srt
_VID_EXTS = ('.mp4', '.mkv', '.webm', '.mov', '.m4v', '.avi')
_LANG_CODES = {k.lower() for k in DEFAULT_VOICES}   # en, es, de, it, ...

# Which folder to sweep:
#   - widget filled              -> that folder
#   - widget empty + multi-lang  -> the whole Dubbing base (every .../<LANG>/ folder)
#   - widget empty + single-lang -> just this run's language folder (.../Dubbing/<LANG>)
# It walks per-language subfolders, reads the language off each subfolder name
# (.../IT/x.mp4 -> 'it'; unknown -> auto-detect) and, with SKIP_EXISTING, fills
# ONLY the videos that don't have an .srt yet — fresh outputs and any older ones.
_src = whisper_source_w.value.strip() if 'whisper_source_w' in globals() else ''
if _src:
    BASE = Path(_src)
elif 'CONFIG' in globals():
    BASE = Path(CONFIG['output_dir'])
    if CONFIG.get('multilang') and BASE.name.upper() in DEFAULT_VOICES:
        BASE = BASE.parent           # .../Dubbing/IT -> .../Dubbing (all languages)
else:
    raise RuntimeError(
        'No folder to sweep. Type the Dubbing folder in the cell above (e.g. '
        '/content/drive/MyDrive/Dubbing or .../Dubbing/DE), re-run it, then run this one.')
if not BASE.is_dir():
    raise RuntimeError(f'Folder to subtitle not found: {BASE}')

def _collect_targets(base):
    pairs = []
    for p in sorted(base.rglob('*')):
        if p.suffix.lower() in _VID_EXTS:
            pairs.append((p, p.parent.name.lower()))
    seen, out = set(), []                # de-dup, keep order
    for p, lg in pairs:
        if p not in seen:
            seen.add(p); out.append((p, lg))
    return out

targets = _collect_targets(BASE)
if not targets:
    raise RuntimeError(f'No videos found under {BASE}.')

if torch.cuda.is_available():
    model = WhisperModel(WHISPER_MODEL, device='cuda', compute_type='int8_float16')
    print(f'Whisper {WHISPER_MODEL} on CUDA (int8_float16)')
else:
    model = WhisperModel(WHISPER_MODEL, device='cpu', compute_type='int8')
    print(f'Whisper {WHISPER_MODEL} on CPU (int8) — slower')

total = len(targets); n_done = n_skip = n_fail = 0
print(f'Sweeping {BASE}')
print(f'{total} video(s) found — filling any that lack an .srt.\n')
for i, (vid, folder_code) in enumerate(targets, 1):
    srt_out = vid.with_suffix('.srt')
    if SKIP_EXISTING and srt_out.exists():
        print(f'[{i}/{total}] (has .srt) {vid.name}'); n_skip += 1; continue
    lang = folder_code if folder_code in _LANG_CODES else None   # None -> auto-detect
    print(f'[{i}/{total}] {vid.name}  [{lang or "auto"}]')
    try:
        segments, info = model.transcribe(
            str(vid), language=lang, task='transcribe',
            temperature=0.0, beam_size=5, best_of=1, vad_filter=WHISPER_VAD,
        )
        cues = []
        for seg in segments:
            txt = (seg.text or '').strip()
            if txt:
                cues.append(srt.Subtitle(
                    index=len(cues) + 1,
                    start=timedelta(seconds=seg.start),
                    end=timedelta(seconds=seg.end),
                    content=txt))
        srt_out.write_text(srt.compose(cues), encoding='utf-8')
        print(f'        -> {srt_out.name}  ({len(cues)} cues, lang={info.language})')
        n_done += 1
    except Exception as e:
        print(f'        x FAILED: {e}'); n_fail += 1

print(f'\nDone. {n_done} transcribed | {n_skip} already had .srt | {n_fail} failed')
print(f'Each .srt is saved next to its video, inside {BASE}.')


## 8. Download outputs (local mode only)

In local mode this pulls each dubbed video **and** its Whisper `.srt` sidecar. In Drive mode they're already saved next to each other in your Drive.

In [ ]:
from google.colab import files
from pathlib import Path
import os
drive_mounted = os.path.ismount('/content/drive')
n_done = n_drive = n_dl = 0
for it in results:
    if it.status != 'done' or not it.output_path: continue
    n_done += 1; p = Path(it.output_path)
    if not p.exists():
        print(f'  [WARN] {p.name} marked done but missing at {p}'); continue
    if drive_mounted and str(p).startswith('/content/drive/'):
        n_drive += 1; print(f'  (skip) {p.name} - already in Drive'); continue
    for f in (p, p.with_suffix('.srt')):      # dubbed video + its Whisper subtitle (step 7)
        if f.exists():
            print(f'  Downloading {f.name}...'); files.download(str(f)); n_dl += 1
print(f'\n{n_done} done | {n_drive} in Drive | {n_dl} files downloaded (video + srt)')


## Free disk: processing cache

Removes the per-video cache (`/tmp/sta-ru-cache`) used to reuse each video's ambient across languages. Run when you're done with a set.

In [ ]:
import shutil, os
for p in ('/tmp/sta-ru-cache', '/tmp/sta-ru-edge', '/tmp/sta-ru-work'):
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True); print(f'Removed {p}')
    else:
        print(f'(skip) {p} not present')
!df -h /tmp | tail -1
